# 🔄 Function Dispatch & Iteration Patterns - Complete Reference

A comprehensive guide to calling functions dynamically, dispatch patterns, decorators, and iteration techniques.

---

## Table of Contents

1. [Why Dynamic Dispatch?](#1-why-dynamic-dispatch)
2. [Dictionary Dispatch (Most Common)](#2-dictionary-dispatch)
3. [getattr() - Dynamic Attribute Access](#3-getattr---dynamic-attribute-access)
4. [Decorator Registry Pattern](#4-decorator-registry-pattern)
5. [Match Statement (Python 3.10+)](#5-match-statement-python-310)
6. [Iteration Patterns](#6-iteration-patterns)
7. [Higher-Order Functions](#7-higher-order-functions)
8. [Function Factories](#8-function-factories)
9. [Partial Functions](#9-partial-functions)
10. [Method Chaining](#10-method-chaining)
11. [Real-World Examples](#11-real-world-examples)

---

## 1. Why Dynamic Dispatch?

The problem: Too many if/elif statements when selecting functions by name.

In [ ]:
# =============================================================================
# THE PROBLEM: Repetitive if/elif chains
# =============================================================================

def grayscale(pixels):
    return "grayscale applied"

def sepia(pixels):
    return "sepia applied"

def blur(pixels):
    return "blur applied"

def invert(pixels):
    return "invert applied"

def sharpen(pixels):
    return "sharpen applied"


# ❌ BAD: Hard to maintain, error-prone, violates DRY
def apply_filter_bad(filter_name: str, pixels: list) -> str:
    if filter_name == "grayscale":
        return grayscale(pixels)
    elif filter_name == "sepia":
        return sepia(pixels)
    elif filter_name == "blur":
        return blur(pixels)
    elif filter_name == "invert":
        return invert(pixels)
    elif filter_name == "sharpen":
        return sharpen(pixels)
    # What if we add 20 more filters?
    else:
        raise ValueError(f"Unknown filter: {filter_name}")


print(apply_filter_bad("sepia", []))  # sepia applied

**Problems with if/elif chains:**
- Repetitive boilerplate
- Easy to make typos
- Hard to add/remove options
- Violates Open/Closed Principle
- No way to iterate over available options

---

## 2. Dictionary Dispatch

The most common and Pythonic solution. Maps names to functions.

In [ ]:
# =============================================================================
# BASIC DICTIONARY DISPATCH
# =============================================================================

from typing import Callable

# Define functions
def grayscale(pixels: list) -> list:
    print("Applying grayscale...")
    return pixels

def sepia(pixels: list) -> list:
    print("Applying sepia...")
    return pixels

def blur(pixels: list) -> list:
    print("Applying blur...")
    return pixels


# ✅ GOOD: Map names to functions
# Keys = string names
# Values = function objects (NOT called - no parentheses!)
FILTERS: dict[str, Callable[[list], list]] = {
    "grayscale": grayscale,
    "sepia": sepia,
    "blur": blur,
}


def apply_filter(filter_name: str, pixels: list) -> list:
    """Apply filter by name using dictionary dispatch."""
    # Step 1: Validate
    if filter_name not in FILTERS:
        available = ", ".join(FILTERS.keys())
        raise ValueError(f"Unknown filter '{filter_name}'. Available: {available}")
    
    # Step 2: Get function from dict
    filter_func = FILTERS[filter_name]
    
    # Step 3: Call and return
    return filter_func(pixels)


# Usage
apply_filter("grayscale", [])  # Applying grayscale...
apply_filter("sepia", [])      # Applying sepia...

In [ ]:
# =============================================================================
# HOW DICTIONARY DISPATCH WORKS (Step by Step)
# =============================================================================

# Functions are first-class objects - they can be stored in variables
def greet(name):
    return f"Hello, {name}!"

# 1. Store function (NOT calling it - no parentheses)
my_func = greet
print(f"Stored: {my_func}")  # <function greet at 0x...>
print(f"Type: {type(my_func)}")  # <class 'function'>

# 2. Call through the variable
result = my_func("Manuel")
print(f"Result: {result}")  # Hello, Manuel!

In [ ]:
# Now with a dictionary
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

# Dictionary maps names to functions
OPERATIONS = {
    "add": add,           # Value is the function object
    "subtract": subtract,
    "multiply": multiply,
}

# Look up by name
operation_name = "multiply"
func = OPERATIONS[operation_name]  # Gets the function object
print(f"Function: {func}")  # <function multiply at 0x...>

# Call it
result = func(5, 3)  # Same as: multiply(5, 3)
print(f"Result: {result}")  # 15

# One-liner: get and call
result = OPERATIONS["add"](10, 20)  # 30
print(f"Add result: {result}")

In [ ]:
# =============================================================================
# DICTIONARY DISPATCH WITH TYPE HINTS
# =============================================================================

from typing import Callable, TypeAlias

# Define type alias for our function signature
# Callable[[param_types], return_type]
FilterFunc: TypeAlias = Callable[[list], list]

def grayscale(pixels: list) -> list:
    """Convert to grayscale."""
    print("Grayscale applied")
    return pixels

def sepia(pixels: list) -> list:
    """Apply sepia tone."""
    print("Sepia applied")
    return pixels

def blur(pixels: list) -> list:
    """Apply blur effect."""
    print("Blur applied")
    return pixels

# Fully typed dictionary
FILTERS: dict[str, FilterFunc] = {
    "grayscale": grayscale,
    "sepia": sepia,
    "blur": blur,
}


def apply_filter(name: str, pixels: list) -> list:
    """Apply named filter to pixels."""
    if name not in FILTERS:
        raise ValueError(f"Unknown: {name}. Available: {list(FILTERS.keys())}")
    
    filter_func: FilterFunc = FILTERS[name]
    return filter_func(pixels)


# Usage
result = apply_filter("sepia", [[128, 128, 128]])

In [ ]:
# =============================================================================
# APPLYING MULTIPLE FILTERS IN SEQUENCE
# =============================================================================

from typing import Callable, TypeAlias

FilterFunc: TypeAlias = Callable[[list], list]

def grayscale(pixels: list) -> list:
    print("1. Grayscale")
    return pixels

def blur(pixels: list) -> list:
    print("2. Blur")
    return pixels

def sharpen(pixels: list) -> list:
    print("3. Sharpen")
    return pixels

FILTERS: dict[str, FilterFunc] = {
    "grayscale": grayscale,
    "blur": blur,
    "sharpen": sharpen,
}


def apply_filters(filter_names: list[str], pixels: list) -> list:
    """Apply multiple filters in sequence."""
    result = pixels
    
    for name in filter_names:
        if name not in FILTERS:
            raise ValueError(f"Unknown filter: {name}")
        result = FILTERS[name](result)
    
    return result


# Apply chain of filters
apply_filters(["grayscale", "blur", "sharpen"], [])
# Output:
# 1. Grayscale
# 2. Blur
# 3. Sharpen

In [ ]:
# =============================================================================
# DICTIONARY DISPATCH WITH DEFAULT/FALLBACK
# =============================================================================

from typing import Callable

def unknown_command(args: list) -> str:
    return f"Unknown command. Args: {args}"

def cmd_help(args: list) -> str:
    return "Available commands: help, version, run"

def cmd_version(args: list) -> str:
    return "Version 1.0.0"

def cmd_run(args: list) -> str:
    return f"Running with args: {args}"


COMMANDS: dict[str, Callable[[list], str]] = {
    "help": cmd_help,
    "version": cmd_version,
    "run": cmd_run,
}


def execute(command: str, args: list) -> str:
    """Execute command with fallback for unknown commands."""
    # .get() returns default if key not found
    handler = COMMANDS.get(command, unknown_command)
    return handler(args)


print(execute("help", []))              # Available commands...
print(execute("version", []))           # Version 1.0.0
print(execute("run", ["--verbose"]))    # Running with args...
print(execute("invalid", ["x", "y"]))   # Unknown command...

In [ ]:
# =============================================================================
# DICTIONARY DISPATCH WITH LAMBDAS
# =============================================================================

# For simple operations, use lambdas directly in the dict
MATH_OPS: dict[str, Callable[[int, int], int]] = {
    "add": lambda a, b: a + b,
    "subtract": lambda a, b: a - b,
    "multiply": lambda a, b: a * b,
    "divide": lambda a, b: a // b,
    "power": lambda a, b: a ** b,
    "modulo": lambda a, b: a % b,
}


def calculate(op: str, a: int, b: int) -> int:
    """Perform math operation."""
    if op not in MATH_OPS:
        raise ValueError(f"Unknown operation: {op}")
    return MATH_OPS[op](a, b)


print(calculate("add", 10, 5))       # 15
print(calculate("multiply", 10, 5))  # 50
print(calculate("power", 2, 8))      # 256

### Dictionary Dispatch Advantages

| Feature | Benefit |
|---------|--------|
| O(1) lookup | Fast, constant time |
| Easy to extend | Just add to dictionary |
| Iterable | `FILTERS.keys()` lists all options |
| Type hints | Full IDE support |
| Testable | Easy to test individual functions |

---

## 3. getattr() - Dynamic Attribute Access

Access class methods by name string.

In [ ]:
# =============================================================================
# BASIC getattr() USAGE
# =============================================================================

class Calculator:
    """Calculator with named operations."""
    
    def add(self, a: int, b: int) -> int:
        return a + b
    
    def subtract(self, a: int, b: int) -> int:
        return a - b
    
    def multiply(self, a: int, b: int) -> int:
        return a * b
    
    def divide(self, a: int, b: int) -> float:
        return a / b


calc = Calculator()

# Normal method call
result = calc.add(5, 3)  # 8

# getattr: get method by name string
method_name = "add"
method = getattr(calc, method_name)  # Gets the bound method
print(f"Method: {method}")  # <bound method Calculator.add...>

# Call the method
result = method(5, 3)  # 8
print(f"Result: {result}")

# One-liner
result = getattr(calc, "multiply")(10, 5)  # 50
print(f"Multiply: {result}")

In [ ]:
# =============================================================================
# getattr() WITH VALIDATION
# =============================================================================

class ImageProcessor:
    """Image processor with filter methods."""
    
    def grayscale(self, pixels: list) -> list:
        print("Grayscale applied")
        return pixels
    
    def sepia(self, pixels: list) -> list:
        print("Sepia applied")
        return pixels
    
    def blur(self, pixels: list) -> list:
        print("Blur applied")
        return pixels
    
    def apply(self, filter_name: str, pixels: list) -> list:
        """Apply filter by name."""
        # Check if method exists
        if not hasattr(self, filter_name):
            raise ValueError(f"Unknown filter: {filter_name}")
        
        # Get and call method
        method = getattr(self, filter_name)
        return method(pixels)


processor = ImageProcessor()
processor.apply("grayscale", [])  # Grayscale applied
processor.apply("sepia", [])      # Sepia applied
# processor.apply("invalid", [])  # ValueError: Unknown filter

In [ ]:
# =============================================================================
# getattr() WITH DEFAULT VALUE
# =============================================================================

class Config:
    debug = True
    timeout = 30
    retries = 3

config = Config()

# Get existing attribute
debug = getattr(config, "debug", False)  # True

# Get non-existing attribute with default
missing = getattr(config, "missing_attr", "default_value")  # default_value

print(f"debug: {debug}")
print(f"missing: {missing}")

In [ ]:
# =============================================================================
# getattr() FOR COMMAND PATTERN
# =============================================================================

class CLI:
    """Command-line interface with subcommands."""
    
    def cmd_help(self, args: list[str]) -> str:
        """Show help message."""
        return "Commands: help, version, run, build"
    
    def cmd_version(self, args: list[str]) -> str:
        """Show version."""
        return "v1.0.0"
    
    def cmd_run(self, args: list[str]) -> str:
        """Run the application."""
        return f"Running with: {args}"
    
    def cmd_build(self, args: list[str]) -> str:
        """Build the project."""
        return f"Building with: {args}"
    
    def execute(self, command: str, args: list[str]) -> str:
        """Execute a command by name."""
        # Add 'cmd_' prefix to prevent accessing other methods
        method_name = f"cmd_{command}"
        
        if not hasattr(self, method_name):
            return f"Unknown command: {command}"
        
        method = getattr(self, method_name)
        return method(args)


cli = CLI()
print(cli.execute("help", []))              # Commands: help...
print(cli.execute("run", ["--verbose"]))    # Running with...
print(cli.execute("invalid", []))           # Unknown command

### Dictionary Dispatch vs getattr()

| Feature | Dictionary Dispatch | getattr() |
|---------|--------------------|-----------|
| Explicit registration | Yes | No (automatic) |
| Control over available functions | Full control | All methods exposed |
| Works with standalone functions | Yes | No (methods only) |
| Security | Better (explicit) | Need prefixing |
| Best for | Module functions | Class methods |

---

## 4. Decorator Registry Pattern

Auto-register functions using decorators.

In [ ]:
# =============================================================================
# BASIC DECORATOR REGISTRY
# =============================================================================

from typing import Callable, TypeAlias

FilterFunc: TypeAlias = Callable[[list], list]

# Registry dictionary (global)
FILTERS: dict[str, FilterFunc] = {}


def register_filter(func: FilterFunc) -> FilterFunc:
    """
    Decorator that registers a function in FILTERS.
    
    Uses the function's name as the key.
    """
    FILTERS[func.__name__] = func
    return func  # Return unchanged function


# Just add decorator - function auto-registers!
@register_filter
def grayscale(pixels: list) -> list:
    """Convert to grayscale."""
    print("Grayscale applied")
    return pixels

@register_filter
def sepia(pixels: list) -> list:
    """Apply sepia tone."""
    print("Sepia applied")
    return pixels

@register_filter
def blur(pixels: list) -> list:
    """Apply blur effect."""
    print("Blur applied")
    return pixels


# See what's registered
print(f"Registered filters: {list(FILTERS.keys())}")
# ['grayscale', 'sepia', 'blur']

# Use it
FILTERS["sepia"]([])  # Sepia applied

In [ ]:
# =============================================================================
# HOW DECORATORS WORK
# =============================================================================

# A decorator is just a function that takes a function and returns a function

def my_decorator(func):
    print(f"Decorating: {func.__name__}")
    return func

# This:
@my_decorator
def greet():
    print("Hello!")

# Is equivalent to:
# def greet():
#     print("Hello!")
# greet = my_decorator(greet)

In [ ]:
# =============================================================================
# DECORATOR REGISTRY WITH CUSTOM NAME
# =============================================================================

from typing import Callable

COMMANDS: dict[str, Callable] = {}


def command(name: str) -> Callable:
    """
    Decorator factory that registers with custom name.
    
    Usage: @command("my-command")
    """
    def decorator(func: Callable) -> Callable:
        COMMANDS[name] = func
        return func
    return decorator


# Register with custom names (can use hyphens, etc.)
@command("show-help")
def help_command():
    return "Showing help..."

@command("list-all")
def list_command():
    return "Listing all items..."

@command("do-something")
def do_something():
    return "Doing something..."


print(f"Commands: {list(COMMANDS.keys())}")
# ['show-help', 'list-all', 'do-something']

print(COMMANDS["show-help"]())  # Showing help...

In [ ]:
# =============================================================================
# DECORATOR REGISTRY WITH METADATA
# =============================================================================

from typing import Callable, TypedDict
from dataclasses import dataclass

@dataclass
class CommandInfo:
    """Metadata about a registered command."""
    func: Callable
    name: str
    description: str
    aliases: list[str]


COMMANDS: dict[str, CommandInfo] = {}


def command(
    name: str,
    description: str = "",
    aliases: list[str] | None = None
) -> Callable:
    """Register command with metadata."""
    def decorator(func: Callable) -> Callable:
        info = CommandInfo(
            func=func,
            name=name,
            description=description or func.__doc__ or "",
            aliases=aliases or []
        )
        
        # Register under main name
        COMMANDS[name] = info
        
        # Register under aliases too
        for alias in info.aliases:
            COMMANDS[alias] = info
        
        return func
    return decorator


@command("help", description="Show help message", aliases=["h", "?"])
def cmd_help():
    return "Help message"

@command("version", description="Show version", aliases=["v", "--version"])
def cmd_version():
    return "v1.0.0"


# Access by name or alias
print(COMMANDS["help"].func())      # Help message
print(COMMANDS["h"].func())         # Help message (alias)
print(COMMANDS["?"].func())         # Help message (alias)

# Access metadata
print(f"Description: {COMMANDS['help'].description}")
print(f"Aliases: {COMMANDS['help'].aliases}")

In [ ]:
# =============================================================================
# CLASS-BASED REGISTRY
# =============================================================================

from typing import Callable, TypeVar

F = TypeVar('F', bound=Callable)


class Registry:
    """Reusable registry for functions."""
    
    def __init__(self, name: str = "registry") -> None:
        self.name = name
        self._items: dict[str, Callable] = {}
    
    def register(self, name: str | None = None) -> Callable[[F], F]:
        """Decorator to register a function."""
        def decorator(func: F) -> F:
            key = name or func.__name__
            self._items[key] = func
            return func
        return decorator
    
    def get(self, name: str) -> Callable:
        """Get function by name."""
        if name not in self._items:
            raise KeyError(f"'{name}' not found in {self.name}")
        return self._items[name]
    
    def __contains__(self, name: str) -> bool:
        return name in self._items
    
    def __iter__(self):
        return iter(self._items)
    
    def items(self):
        return self._items.items()


# Create separate registries
filters = Registry("filters")
validators = Registry("validators")


@filters.register()
def grayscale(pixels):
    return "grayscale"

@filters.register("blur-3x3")
def blur(pixels):
    return "blur"

@validators.register()
def check_size(image):
    return True


print(f"Filters: {list(filters)}")
print(f"Validators: {list(validators)}")

print(filters.get("grayscale")([]))  # grayscale
print(filters.get("blur-3x3")([]))   # blur

---

## 5. Match Statement (Python 3.10+)

Structural pattern matching - more powerful than if/elif.

In [ ]:
# =============================================================================
# BASIC MATCH STATEMENT
# =============================================================================

def handle_command(command: str) -> str:
    """Handle command using match statement."""
    match command:
        case "start":
            return "Starting..."
        case "stop":
            return "Stopping..."
        case "restart":
            return "Restarting..."
        case "status":
            return "Running"
        case _:  # Default case (wildcard)
            return f"Unknown command: {command}"


print(handle_command("start"))    # Starting...
print(handle_command("status"))   # Running
print(handle_command("invalid"))  # Unknown command: invalid

In [ ]:
# =============================================================================
# MATCH WITH OR PATTERNS
# =============================================================================

def categorize(value: str | int) -> str:
    match value:
        case "yes" | "y" | "true" | 1:
            return "Affirmative"
        case "no" | "n" | "false" | 0:
            return "Negative"
        case _:
            return "Unknown"


print(categorize("yes"))    # Affirmative
print(categorize("y"))      # Affirmative
print(categorize(1))        # Affirmative
print(categorize("no"))     # Negative
print(categorize("maybe"))  # Unknown

In [1]:
# =============================================================================
# MATCH WITH SEQUENCE PATTERNS
# =============================================================================

def process_args(args: list[str]) -> str:
    match args:
        case []:
            return "No arguments"
        case [cmd]:
            return f"Command: {cmd}"
        case [cmd, arg]:
            return f"Command: {cmd}, Arg: {arg}"
        case [cmd, *rest]:  # Capture remaining
            return f"Command: {cmd}, Rest: {rest}"


print(process_args([]))                      # No arguments
print(process_args(["help"]))                # Command: help
print(process_args(["run", "app.py"]))       # Command: run, Arg: app.py
print(process_args(["run", "a", "b", "c"]))  # Command: run, Rest: ['a', 'b', 'c']

No arguments
Command: help
Command: run, Arg: app.py
Command: run, Rest: ['a', 'b', 'c']


In [ ]:
# =============================================================================
# MATCH WITH DICTIONARY PATTERNS
# =============================================================================

def handle_event(event: dict) -> str:
    match event:
        case {"type": "click", "button": button}:
            return f"Click event: {button} button"
        case {"type": "keypress", "key": key}:
            return f"Key pressed: {key}"
        case {"type": "scroll", "direction": dir, "amount": amt}:
            return f"Scroll {dir} by {amt}"
        case {"type": event_type}:
            return f"Unknown event type: {event_type}"
        case _:
            return "Invalid event"


print(handle_event({"type": "click", "button": "left"}))
# Click event: left button

print(handle_event({"type": "keypress", "key": "Enter"}))
# Key pressed: Enter

print(handle_event({"type": "scroll", "direction": "up", "amount": 100}))
# Scroll up by 100

In [ ]:
# =============================================================================
# MATCH WITH GUARDS (if conditions)
# =============================================================================

def classify_number(n: int) -> str:
    match n:
        case 0:
            return "Zero"
        case x if x < 0:
            return f"Negative: {x}"
        case x if x % 2 == 0:
            return f"Positive even: {x}"
        case x:
            return f"Positive odd: {x}"


print(classify_number(0))    # Zero
print(classify_number(-5))   # Negative: -5
print(classify_number(10))   # Positive even: 10
print(classify_number(7))    # Positive odd: 7

---

## 6. Iteration Patterns

Best practices for iterating over collections.

In [ ]:
# =============================================================================
# BASIC ITERATION
# =============================================================================

names = ["Alice", "Bob", "Charlie"]

# ❌ Bad: Using index
for i in range(len(names)):
    print(names[i])

print("---")

# ✅ Good: Direct iteration
for name in names:
    print(name)

In [ ]:
# =============================================================================
# enumerate() - Index + Value
# =============================================================================

names = ["Alice", "Bob", "Charlie"]

# ❌ Bad: Manual counter
i = 0
for name in names:
    print(f"{i}: {name}")
    i += 1

print("---")

# ✅ Good: enumerate
for i, name in enumerate(names):
    print(f"{i}: {name}")

print("---")

# Start from different index
for i, name in enumerate(names, start=1):
    print(f"{i}: {name}")

In [ ]:
# =============================================================================
# zip() - Parallel Iteration
# =============================================================================

names = ["Alice", "Bob", "Charlie"]
ages = [25, 30, 35]
cities = ["NYC", "LA", "Chicago"]

# ❌ Bad: Index-based
for i in range(len(names)):
    print(f"{names[i]} is {ages[i]} from {cities[i]}")

print("---")

# ✅ Good: zip
for name, age, city in zip(names, ages, cities):
    print(f"{name} is {age} from {city}")

In [ ]:
# =============================================================================
# zip() WITH STRICT MODE (Python 3.10+)
# =============================================================================

# Regular zip stops at shortest
names = ["Alice", "Bob", "Charlie"]
ages = [25, 30]  # Shorter!

for name, age in zip(names, ages):
    print(f"{name}: {age}")
# Only prints 2 items, silently drops Charlie

print("---")

# strict=True raises error if lengths differ
try:
    for name, age in zip(names, ages, strict=True):
        print(f"{name}: {age}")
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# =============================================================================
# Dictionary Iteration
# =============================================================================

scores = {"Alice": 95, "Bob": 87, "Charlie": 92}

# Keys only
for name in scores:
    print(name)

print("---")

# Keys explicitly
for name in scores.keys():
    print(name)

print("---")

# Values only
for score in scores.values():
    print(score)

print("---")

# ✅ Best: Key-value pairs
for name, score in scores.items():
    print(f"{name}: {score}")

In [ ]:
# =============================================================================
# reversed() - Reverse Iteration
# =============================================================================

items = [1, 2, 3, 4, 5]

# ❌ Bad: Create reversed copy
for item in items[::-1]:
    print(item)

print("---")

# ✅ Good: reversed() is memory efficient (no copy)
for item in reversed(items):
    print(item)

In [ ]:
# =============================================================================
# sorted() - Sorted Iteration
# =============================================================================

scores = {"Charlie": 92, "Alice": 95, "Bob": 87}

# Sort by key (name)
for name in sorted(scores):
    print(f"{name}: {scores[name]}")

print("---")

# Sort by value (score)
for name, score in sorted(scores.items(), key=lambda x: x[1]):
    print(f"{name}: {score}")

print("---")

# Sort by value (descending)
for name, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score}")

In [ ]:
# =============================================================================
# itertools - Advanced Iteration
# =============================================================================

from itertools import chain, islice, cycle, repeat, groupby

# chain - combine iterables
list1 = [1, 2, 3]
list2 = [4, 5, 6]
for item in chain(list1, list2):
    print(item, end=" ")  # 1 2 3 4 5 6
print()

# islice - slice iterator
for item in islice(range(100), 5, 10):
    print(item, end=" ")  # 5 6 7 8 9
print()

# cycle - infinite cycling (use islice to limit)
colors = ["red", "green", "blue"]
for color in islice(cycle(colors), 7):
    print(color, end=" ")  # red green blue red green blue red
print()

# repeat - repeat value
for x in repeat("hello", 3):
    print(x, end=" ")  # hello hello hello

In [ ]:
# =============================================================================
# groupby - Group consecutive items
# =============================================================================

from itertools import groupby

# Data must be sorted by grouping key!
data = [
    ("fruit", "apple"),
    ("fruit", "banana"),
    ("vegetable", "carrot"),
    ("vegetable", "potato"),
    ("fruit", "orange"),  # Another fruit group!
]

# Sort first!
data_sorted = sorted(data, key=lambda x: x[0])

for category, items in groupby(data_sorted, key=lambda x: x[0]):
    items_list = [item[1] for item in items]
    print(f"{category}: {items_list}")
    
# fruit: ['apple', 'banana', 'orange']
# vegetable: ['carrot', 'potato']

---

## 7. Higher-Order Functions

Functions that take or return other functions.

In [ ]:
# =============================================================================
# map() - Transform each item
# =============================================================================

numbers = [1, 2, 3, 4, 5]

# map returns iterator, convert to list to see results
squared = list(map(lambda x: x ** 2, numbers))
print(f"Squared: {squared}")  # [1, 4, 9, 16, 25]

# With named function
def double(x):
    return x * 2

doubled = list(map(double, numbers))
print(f"Doubled: {doubled}")  # [2, 4, 6, 8, 10]

# ✅ Often, list comprehension is clearer
squared = [x ** 2 for x in numbers]
doubled = [x * 2 for x in numbers]

In [ ]:
# =============================================================================
# filter() - Select items
# =============================================================================

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Filter even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(f"Evens: {evens}")  # [2, 4, 6, 8, 10]

# Filter with named function
def is_positive(x):
    return x > 0

values = [-2, -1, 0, 1, 2, 3]
positives = list(filter(is_positive, values))
print(f"Positives: {positives}")  # [1, 2, 3]

# ✅ List comprehension equivalent
evens = [x for x in numbers if x % 2 == 0]
positives = [x for x in values if x > 0]

In [ ]:
# =============================================================================
# reduce() - Accumulate to single value
# =============================================================================

from functools import reduce

numbers = [1, 2, 3, 4, 5]

# Sum all numbers
total = reduce(lambda acc, x: acc + x, numbers)
print(f"Sum: {total}")  # 15

# Product of all numbers
product = reduce(lambda acc, x: acc * x, numbers)
print(f"Product: {product}")  # 120

# Find maximum
maximum = reduce(lambda acc, x: acc if acc > x else x, numbers)
print(f"Max: {maximum}")  # 5

# ✅ Often, built-in functions are clearer
print(f"Sum: {sum(numbers)}")
print(f"Max: {max(numbers)}")

In [ ]:
# =============================================================================
# any() and all() - Boolean aggregation
# =============================================================================

numbers = [1, 2, 3, 4, 5]

# any: True if ANY item is truthy
has_positive = any(x > 0 for x in numbers)
print(f"Has positive: {has_positive}")  # True

# all: True if ALL items are truthy
all_positive = all(x > 0 for x in numbers)
print(f"All positive: {all_positive}")  # True

mixed = [-1, 0, 1, 2]
print(f"Any positive: {any(x > 0 for x in mixed)}")  # True
print(f"All positive: {all(x > 0 for x in mixed)}")  # False

---

## 8. Function Factories

Functions that create and return other functions.

In [ ]:
# =============================================================================
# BASIC FUNCTION FACTORY
# =============================================================================

from typing import Callable

def create_multiplier(factor: int) -> Callable[[int], int]:
    """
    Factory that creates multiplier functions.
    
    The inner function "closes over" the factor variable.
    This is called a closure.
    """
    def multiplier(x: int) -> int:
        return x * factor  # factor is captured from outer scope
    
    return multiplier


# Create specialized functions
double = create_multiplier(2)
triple = create_multiplier(3)
quadruple = create_multiplier(4)

print(double(5))     # 10
print(triple(5))     # 15
print(quadruple(5))  # 20

In [ ]:
# =============================================================================
# VALIDATOR FACTORY
# =============================================================================

from typing import Callable

def create_range_validator(
    min_val: int,
    max_val: int
) -> Callable[[int], bool]:
    """Create a validator for a specific range."""
    def validate(value: int) -> bool:
        return min_val <= value <= max_val
    return validate


# Create validators for different ranges
is_valid_age = create_range_validator(0, 150)
is_valid_percentage = create_range_validator(0, 100)
is_valid_rgb = create_range_validator(0, 255)

print(is_valid_age(25))           # True
print(is_valid_age(200))          # False
print(is_valid_percentage(75))    # True
print(is_valid_rgb(255))          # True

In [ ]:
# =============================================================================
# FILTER FACTORY FOR IMAGE PROCESSING
# =============================================================================

from typing import Callable, TypeAlias

Pixel: TypeAlias = list[int]  # [B, G, R]
FilterFunc: TypeAlias = Callable[[Pixel], Pixel]


def create_brightness_filter(adjustment: int) -> FilterFunc:
    """Create a brightness adjustment filter."""
    def adjust_brightness(pixel: Pixel) -> Pixel:
        return [
            max(0, min(255, channel + adjustment))
            for channel in pixel
        ]
    return adjust_brightness


def create_channel_filter(channel_index: int) -> FilterFunc:
    """Create a filter that isolates one color channel."""
    def isolate_channel(pixel: Pixel) -> Pixel:
        return [
            pixel[i] if i == channel_index else 0
            for i in range(3)
        ]
    return isolate_channel


# Create specific filters
brighten = create_brightness_filter(50)
darken = create_brightness_filter(-50)
red_only = create_channel_filter(2)   # R is index 2 in BGR
blue_only = create_channel_filter(0)  # B is index 0 in BGR

pixel = [100, 150, 200]  # BGR
print(f"Original: {pixel}")
print(f"Brightened: {brighten(pixel)}")
print(f"Darkened: {darken(pixel)}")
print(f"Red only: {red_only(pixel)}")

---

## 9. Partial Functions

Pre-fill some function arguments.

In [ ]:
# =============================================================================
# functools.partial - Freeze arguments
# =============================================================================

from functools import partial

def power(base: int, exponent: int) -> int:
    """Calculate base raised to exponent."""
    return base ** exponent


# Create specialized versions
square = partial(power, exponent=2)  # Always squares
cube = partial(power, exponent=3)    # Always cubes

print(square(5))  # 25 (5^2)
print(cube(5))    # 125 (5^3)

In [ ]:
# =============================================================================
# partial FOR API CALLS
# =============================================================================

from functools import partial

def api_call(endpoint: str, method: str, data: dict | None = None) -> str:
    """Simulate API call."""
    return f"{method} {endpoint} with {data}"


# Create specialized API functions
get_users = partial(api_call, "/users", "GET")
post_user = partial(api_call, "/users", "POST")
delete_user = partial(api_call, method="DELETE")

print(get_users())
# GET /users with None

print(post_user(data={"name": "Alice"}))
# POST /users with {'name': 'Alice'}

print(delete_user("/users/123"))
# DELETE /users/123 with None

In [ ]:
# =============================================================================
# partial IN DICTIONARY DISPATCH
# =============================================================================

from functools import partial
from typing import Callable

def adjust_brightness(pixels: list, amount: int) -> list:
    """Adjust brightness by amount."""
    print(f"Adjusting brightness by {amount}")
    return pixels

def apply_blur(pixels: list, radius: int) -> list:
    """Apply blur with given radius."""
    print(f"Applying blur with radius {radius}")
    return pixels


# Create pre-configured filters using partial
FILTERS: dict[str, Callable[[list], list]] = {
    "brighten": partial(adjust_brightness, amount=50),
    "darken": partial(adjust_brightness, amount=-50),
    "blur-light": partial(apply_blur, radius=1),
    "blur-heavy": partial(apply_blur, radius=5),
}


def apply_filter(name: str, pixels: list) -> list:
    return FILTERS[name](pixels)


apply_filter("brighten", [])    # Adjusting brightness by 50
apply_filter("blur-heavy", [])  # Applying blur with radius 5

---

## 10. Method Chaining

Fluent interface pattern for readable code.

In [ ]:
# =============================================================================
# METHOD CHAINING PATTERN
# =============================================================================

class ImageProcessor:
    """Image processor with chainable methods."""
    
    def __init__(self, pixels: list) -> None:
        self.pixels = pixels
        self._operations: list[str] = []
    
    def grayscale(self) -> "ImageProcessor":
        """Apply grayscale."""
        self._operations.append("grayscale")
        return self  # Return self for chaining!
    
    def blur(self, radius: int = 1) -> "ImageProcessor":
        """Apply blur."""
        self._operations.append(f"blur({radius})")
        return self
    
    def brightness(self, amount: int) -> "ImageProcessor":
        """Adjust brightness."""
        self._operations.append(f"brightness({amount})")
        return self
    
    def resize(self, width: int, height: int) -> "ImageProcessor":
        """Resize image."""
        self._operations.append(f"resize({width}x{height})")
        return self
    
    def save(self, path: str) -> None:
        """Save (end of chain)."""
        print(f"Applied: {' -> '.join(self._operations)}")
        print(f"Saved to: {path}")


# Fluent chain of operations
(
    ImageProcessor([[128, 128, 128]])
    .grayscale()
    .blur(radius=3)
    .brightness(50)
    .resize(800, 600)
    .save("output.bmp")
)

# Output:
# Applied: grayscale -> blur(3) -> brightness(50) -> resize(800x600)
# Saved to: output.bmp

---

## 11. Real-World Examples

In [ ]:
# =============================================================================
# COMPLETE BMP FILTER SYSTEM
# =============================================================================

from typing import Callable, TypeAlias
from dataclasses import dataclass
from pathlib import Path

# Type definitions
Pixel: TypeAlias = list[int]           # [B, G, R]
PixelRow: TypeAlias = list[Pixel]
ImageData: TypeAlias = list[PixelRow]
FilterFunc: TypeAlias = Callable[[ImageData], ImageData]


@dataclass
class FilterInfo:
    """Metadata about a filter."""
    func: FilterFunc
    name: str
    description: str


# Filter registry
FILTERS: dict[str, FilterInfo] = {}


def register_filter(name: str, description: str = "") -> Callable[[FilterFunc], FilterFunc]:
    """Decorator to register a filter."""
    def decorator(func: FilterFunc) -> FilterFunc:
        FILTERS[name] = FilterInfo(
            func=func,
            name=name,
            description=description or func.__doc__ or ""
        )
        return func
    return decorator


# Register filters
@register_filter("grayscale", "Convert to grayscale using luminosity")
def grayscale(pixels: ImageData) -> ImageData:
    """Convert to grayscale."""
    result = []
    for row in pixels:
        new_row = []
        for b, g, r in row:
            gray = int(0.299 * r + 0.587 * g + 0.114 * b)
            new_row.append([gray, gray, gray])
        result.append(new_row)
    return result


@register_filter("invert", "Invert all colors")
def invert(pixels: ImageData) -> ImageData:
    """Invert colors."""
    return [[[255 - c for c in pixel] for pixel in row] for row in pixels]


@register_filter("sepia", "Apply sepia tone")
def sepia(pixels: ImageData) -> ImageData:
    """Apply sepia tone."""
    result = []
    for row in pixels:
        new_row = []
        for b, g, r in row:
            new_r = min(255, int(0.393*r + 0.769*g + 0.189*b))
            new_g = min(255, int(0.349*r + 0.686*g + 0.168*b))
            new_b = min(255, int(0.272*r + 0.534*g + 0.131*b))
            new_row.append([new_b, new_g, new_r])
        result.append(new_row)
    return result


# Application functions
def list_filters() -> list[str]:
    """List all available filters."""
    return list(FILTERS.keys())


def get_filter_help(name: str) -> str:
    """Get help for a specific filter."""
    if name not in FILTERS:
        return f"Unknown filter: {name}"
    info = FILTERS[name]
    return f"{info.name}: {info.description}"


def apply_filter(name: str, pixels: ImageData) -> ImageData:
    """Apply a single filter."""
    if name not in FILTERS:
        raise ValueError(f"Unknown filter: {name}. Available: {list_filters()}")
    return FILTERS[name].func(pixels)


def apply_filters(names: list[str], pixels: ImageData) -> ImageData:
    """Apply multiple filters in sequence."""
    result = pixels
    for name in names:
        result = apply_filter(name, result)
    return result


# Usage
print(f"Available filters: {list_filters()}")
print(f"Help: {get_filter_help('grayscale')}")

# Test with sample data
test_pixels = [[[100, 150, 200], [50, 100, 150]]]
result = apply_filters(["grayscale", "invert"], test_pixels)
print(f"Original: {test_pixels}")
print(f"Filtered: {result}")

---

## 📋 Quick Reference

```python
# =============================================================================
# FUNCTION DISPATCH PATTERNS QUICK REFERENCE
# =============================================================================
#
# DICTIONARY DISPATCH:
#     FUNCS = {"name": func}
#     FUNCS["name"](args)
#
# getattr():
#     method = getattr(obj, "method_name")
#     method(args)
#
# DECORATOR REGISTRY:
#     REGISTRY = {}
#     def register(func):
#         REGISTRY[func.__name__] = func
#         return func
#
# MATCH STATEMENT (Python 3.10+):
#     match value:
#         case "x": ...
#         case _: ...  # default
#
# ITERATION:
#     for item in items:           # Direct
#     for i, item in enumerate():  # With index
#     for a, b in zip(xs, ys):     # Parallel
#     for k, v in dict.items():    # Dict pairs
#
# HIGHER-ORDER FUNCTIONS:
#     map(func, items)     # Transform
#     filter(func, items)  # Select
#     reduce(func, items)  # Accumulate
#
# PARTIAL:
#     from functools import partial
#     double = partial(multiply, factor=2)
#
# =============================================================================
```